# ch01 练习：数据划分与流水线

与讲解版逐 Cell 对应。核心代码被挖空，`assert` 验收**保留**——
跑通所有 `assert` 即自查通过。卡住就回讲解版看对应小节。

## 一、考点回顾

| 竞赛评分点 | 本节覆盖 |
|---|---|
| 模型选择 10% | 划分 + 三种 CV 选型 |
| 模型训练 40% | Pipeline + ColumnTransformer 组装 |

## 二、API 速查

| 方法 | 关键参数 | 一句话 |
|---|---|---|
| `train_test_split` | `test_size` `random_state` `stratify` | 保留法划分 |
| `StratifiedKFold` | `n_splits` `shuffle` `random_state` | 分类折外验证 |
| `TimeSeriesSplit` | `n_splits` `test_size` `gap` | 时序折外验证 |
| `Pipeline` | `[(名, 步骤)]` | 顺序执行防泄漏 |
| `ColumnTransformer` | `transformers` `remainder` | 按列预处理 |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (KFold, StratifiedKFold, TimeSeriesSplit,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

DATA = Path("data")

dev = pd.read_csv(DATA / "defect_ml.csv")   # 分类：预测「是否危急」
reg = pd.read_csv(DATA / "load_reg.csv")    # 回归：预测「日均负荷」（后面章节用）

NUM = ["负荷值", "温度", "湿度", "投运年限", "巡检耗时分钟"]
CAT = ["设备类型", "缺陷类型"]
TARGET = "是否危急"

X = dev[NUM + CAT]
y = dev[TARGET]

print("sklearn", sklearn.__version__)
print("dev", dev.shape, "| 危急", int(y.sum()), f"({y.mean():.1%})")
print("reg", reg.shape)

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。

    返回 ("ok", 结果) 或 ("err", 异常类型名)。
    
"""
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


def fold_report(tag, splits, y_series):
    """打印每一折的 (train 行数, test 行数, test 正例数, test 正例率)。
"""
    for i, (tr, te) in enumerate(splits):
        pos = int(y_series.iloc[te].sum())
        print(f"  {tag} fold{i}: train={len(tr)} test={len(te)} "
              f"test正例={pos} 正例率={pos / len(te):.4f}")


def make_ct(remainder="drop"):
    """统一的 ColumnTransformer：数值 = 中位数填充 + 标准化；类别 = 众数填充 + OneHot。
"""
    num_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([("num", num_pipe, NUM), ("cat", cat_pipe, CAT)],
                             remainder=remainder)


# 演示 TimeSeriesSplit 用的时序数据：365 天日负荷（干净，只为看折结构）
TS = pd.DataFrame({"t": np.arange(365),
                   "负荷": np.random.default_rng(42).normal(400, 50, 365)})

print("TS", TS.shape)

## 题 1：保留法划分（对应讲解 §3.1）

In [ ]:
# TODO(1)：按 8:2 切分 X, y，固定随机种子 42，得到 X_tr, X_te, y_tr, y_te
# 提示：train_test_split(X, y, test_size=..., random_state=...)
X_tr, X_te, y_tr, y_te = ____

# 切两次必须一致（可复现是评分硬要求）
again = train_test_split(X, y, test_size=0.2, random_state=42)
print("形状:", X_tr.shape, X_te.shape, "| 索引前 6:", X_tr.index[:6].tolist())
print("可复现:", bool(again[0].index.equals(X_tr.index)))

assert X_tr.shape == (480, 7) and X_te.shape == (120, 7)
assert int(y_tr.sum()) == 108 and int(y_te.sum()) == 20
assert X_tr.index[:6].tolist() == [145, 9, 375, 523, 188, 131]

**为什么这样做**：`random_state=42` 让评分官重跑你的代码得到一模一样的划分；
不写这个参数，两次运行连你自己都对不上。

## 题 2：不平衡数据的分层切分（对应讲解 §3.2）

In [ ]:
# TODO(2)：在题 1 基础上加分层参数，得到 Xs_tr, Xs_te, ys_tr, ys_te
# 提示：stratify 接的是标签 y
Xs_tr, Xs_te, ys_tr, ys_te = ____

print("分层后：训练正例", int(ys_tr.sum()), f"({ys_tr.mean():.4f})",
      "| 测试正例", int(ys_te.sum()), f"({ys_te.mean():.4f})")

assert int(ys_tr.sum()) == 102 and int(ys_te.sum()) == 26
assert round(ys_tr.mean(), 4) == 0.2125 and round(ys_te.mean(), 4) == 0.2167

## 题 3：StratifiedKFold vs KFold（对应讲解 §3.4）

In [ ]:
# TODO(3)：定义 5 折的 StratifiedKFold（shuffle=True，种子 42），并生成折列表
# 提示：StratifiedKFold(n_splits=..., shuffle=..., random_state=...) 的 .split(X, y)
skf = ____
strat_splits = ____

# TODO(4)：再定义一个普通 KFold（同样 5 折、shuffle、种子 42），生成折列表
kf = ____
kfold_splits = ____

skf_pos = [int(y.iloc[te].sum()) for _, te in strat_splits]
kf_pos = [int(y.iloc[te].sum()) for _, te in kfold_splits]
print("strat 各折正例:", skf_pos)
print("kfold 各折正例:", kf_pos, "← 折间跳变，分层缺位")

assert skf_pos == [25, 25, 26, 26, 26]
assert kf_pos == [20, 21, 34, 27, 26]

## 题 4：TimeSeriesSplit 折结构（对应讲解 §3.5）

In [ ]:
# TODO(5)：对 TS 定义 5 折的 TimeSeriesSplit，并生成折列表
# 提示：TimeSeriesSplit(n_splits=...) 的 .split(TS)
tss = ____
ts_splits = ____

train_sizes = [len(tr) for tr, _ in ts_splits]
test_sizes = [len(te) for _, te in ts_splits]
print("train:", train_sizes, "| test:", test_sizes)

assert train_sizes == [65, 125, 185, 245, 305]
assert test_sizes == [60] * 5
assert ts_splits[0][1][0] == 65

**为什么这样做**：时序折的 train 只能来自 test 的过去。索引 0..64 训练、
65..124 验证，才符合"上线时模型只见过历史"的真实场景。

## 题 5：ColumnTransformer 组装与 remainder 陷阱（对应讲解 §3.6-3.7）

In [ ]:
# TODO(6)：复用脚手架的 make_ct()：对 X 做变换，结果存入 Xt
ct = ____
Xt = ____

print("变换后形状:", Xt.shape)
assert Xt.shape == (600, 16)
assert len(ct.get_feature_names_out()) == 16
assert ct.get_feature_names_out()[0] == "num__负荷值"

# TODO(7)：给 X 加一列「备注」后用默认 make_ct() 变换，观察是否被静默丢弃
# 提示：X.assign(备注=[...] * len(X))
X3 = ____
Xt_drop = ____
Xt_pass = ____

print("默认:", Xt_drop.shape, "| passthrough:", Xt_pass.shape)
assert Xt_drop.shape == (600, 16), "默认 drop：备注列静默消失"
assert Xt_pass.shape == (600, 17), "passthrough：多出一列"

## 题 6：未知类别与全流程流水线（对应讲解 §3.9-3.10）

In [ ]:
# TODO(8)：定义 handle_unknown="ignore" 的 OneHotEncoder（非稀疏输出），
#            在 X[['设备类型']] 上 fit，并对 ['杆塔', '变压器'] 做变换存入 out
# 提示：OneHotEncoder(handle_unknown=..., sparse_output=False)
oh = ____
____
out = ____

print("学到的类别:", oh.categories_[0].tolist())
print("未知类别行的独热和:", float(out[0].sum()))
assert oh.categories_[0].tolist() == ["互感器", "变压器", "断路器", "绝缘子", "避雷器"]
assert out.shape == (2, 5) and float(out[0].sum()) == 0.0

In [ ]:
# TODO(9)：把 make_ct() 与 LogisticRegression(max_iter=1000) 装进一条 Pipeline
# 提示：Pipeline([("ct", ...), ("clf", ...)])
pipe = ____

# TODO(10)：用分层训练段 fit，在分层测试段上 score，结果存入 acc
____
acc = ____

pred = pipe.predict(Xs_te)
print("test accuracy:", round(float(acc), 4))
print("预测正例", int(pred.sum()), "vs 实际", int(ys_te.sum()), "← 阈值现象，ch07 见")

assert round(float(acc), 4) == 0.7917
assert int(pred.sum()) == 9 and int(ys_te.sum()) == 26

---

## 综合自查

1. 题 1、2 的差别只有 `stratify=y` 一个参数——为什么占比从漂移变成锁定？
2. 题 3 两种分割器差在哪？什么数据必须用第三种？
3. 题 7 里被丢掉的列，怎么改参数能保住？
4. 题 9 的 Pipeline 为什么天然防泄漏？

全部答得上来，进入 ch02（特征工程）。